# Phase 2 — Nettoyer et découper les données

**Objectif** : passer des avis bruts (`data/raw/`) à trois jeux propres et prêts pour la
modélisation : **train**, **validation** et **test**. À la fin de ce notebook, chaque règle de
nettoyage est justifiée, mesurée (combien d'avis elle touche) et vérifiée.

**Avant de commencer**

1. Les données brutes doivent exister : `uv run python -m review_pilot.data.download` (depuis la racine du projet).
2. Dans VS Code, choisis le noyau (*kernel*) `.venv` en haut à droite du notebook.

**Méthode** : chaque section pose ses questions, puis donne le code, les résultats et une
cellule « Observations ». Les blocs repliés **📖 Comprendre** expliquent les notions utilisées.

> Règle d'or : les règles de nettoyage se **décident en regardant le train uniquement**.
> On les applique ensuite telles quelles au test, dont on ne regarde que des **comptes**,
> jamais le contenu des avis.

## 0. Le contexte : que fait-on dans cette phase ?

**Où en est le projet.** La phase 1 a fixé le besoin : **signaler au service client les avis
négatifs**, en en trouvant au moins 90 % (rappel) avec au plus ~15 % de fausses alertes.
L'EDA (`01_eda.ipynb`) a dressé la liste des problèmes des données. Cette phase les corrige.

**Pourquoi nettoyer ?** Un modèle apprend ce qu'on lui montre. Des avis en espagnol, des restes
de code HTML ou des phrases collées sont du « bruit » : au mieux inutile, au pire trompeur.
Et des données personnelles (e-mails, téléphones) n'ont rien à faire dans un jeu d'entraînement.

**Les trois jeux de données produits**

| Jeu | Rôle | Analogie |
|---|---|---|
| **train** | le modèle apprend dessus | les exercices du cours |
| **validation** | on compare les modèles et on règle leurs paramètres dessus | les examens blancs |
| **test** | la note finale, utilisée **une seule fois** (phase 5) | l'examen final |

**Ce que la phase apporte au projet.** Des données propres, et surtout un procédé
**reproductible** : une fois ce notebook validé, ses règles seront recopiées dans le code du
projet (`src/review_pilot/data/`), testées, et une seule commande recréera les trois jeux.

<details>
<summary>📚 Lexique — les mots techniques de ce notebook</summary>

| Mot | Sens |
|---|---|
| **Nettoyage** | Corriger ou retirer ce qui abîme les données (bruit, erreurs, doublons). |
| **Règle déterministe** | Une règle qui donne toujours le même résultat pour le même texte, sans rien « apprendre » des données. |
| **Expression régulière** (regex) | Une mini-syntaxe pour décrire un motif dans un texte (« une minuscule suivie d'un point »). |
| **Entité HTML** | Un code qui remplace un caractère spécial dans une page web : `&amp;` pour `&`, `&quot;` pour `"`. |
| **Balise HTML** | Une instruction de mise en page entre chevrons : `<i>` (italique), `<br />` (retour à la ligne). |
| **Heuristique** | Une règle simple et approximative, utilisée quand une méthode exacte serait trop coûteuse. |
| **Faux positif** | Un cas qu'une règle attrape à tort (une référence de produit prise pour un code postal). |
| **Donnée personnelle** | Une information qui permet d'identifier ou de contacter quelqu'un (e-mail, téléphone, adresse). |
| **Masquage** | Remplacer une donnée personnelle par un repère neutre (`[EMAIL]`) au lieu de supprimer tout l'avis. |
| **Doublon / quasi-doublon** | Deux avis identiques / identiques à la ponctuation et aux majuscules près. |
| **Fuite de données** | Quand un avis du test se retrouve aussi dans le train : la note finale serait faussée. |
| **Validation** (jeu de) | Une partie du train mise de côté pour comparer et régler les modèles sans toucher au test. |
| **Découpage stratifié** | Un découpage qui garde la même proportion de chaque label dans chaque jeu. |
| **Graine** (seed) | Un nombre qui rend un tirage « aléatoire » répétable à l'identique. |
| **Contrôle qualité** | Une vérification automatique (« aucun texte vide ») qui doit toujours être vraie. |
| **Pipeline** | L'enchaînement des étapes (charger → nettoyer → découper → vérifier), relançable d'une commande. |

</details>

## 1. Imports, chargement et journal des étapes

- Quelles librairies faut-il pour nettoyer du texte et découper les données ?
- Les données brutes sont-elles bien celles de l'EDA (50 000 avis train, 10 000 test) ?
- Comment garder la trace de ce que chaque règle retire ou modifie ?

<details>
<summary>📖 Comprendre</summary>

- On garde une copie **intacte** des données brutes (`raw_train`, `raw_test`) : à la fin, on
  pourra comparer « avant » et « après » et expliquer chaque ligne perdue.
- Le **journal** (`cleaning_log`) note, pour chaque étape et chaque jeu : le nombre de lignes
  avant, après, et le nombre de textes modifiés. C'est lui qui produira le bilan de la section 9.
- `re` sert aux expressions régulières, `html` à décoder les entités HTML, et `scikit-learn`
  fournit la fonction de découpage stratifié (section 7).

</details>

In [ ]:
import html
import re
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

pd.set_option("display.max_colwidth", 120)

# Le notebook s'exécute depuis notebooks/ : les données sont un dossier plus haut.
RAW_DIR = Path("../data/raw")
SEED = 42

In [ ]:
raw_train = pd.read_parquet(RAW_DIR / "train.parquet")
raw_test = pd.read_parquet(RAW_DIR / "test.parquet")

# Copies de travail : les données brutes restent intactes pour le bilan final.
train = raw_train.copy()
test = raw_test.copy()
print("train :", train.shape, "| test :", test.shape)

In [ ]:
cleaning_log: list[dict[str, object]] = []


def log_step(step: str, split: str, rows_before: int, rows_after: int, changed: int) -> None:
    """Note dans le journal l'effet d'une étape sur un jeu de données."""
    cleaning_log.append(
        {
            "étape": step,
            "jeu": split,
            "lignes avant": rows_before,
            "lignes après": rows_after,
            "textes modifiés": changed,
        }
    )

**Observations** : 50 000 avis dans le train et 10 000 dans le test, 3 colonnes chacun : ce sont
bien les données de l'EDA. Le journal est vide pour l'instant ; chaque étape y ajoutera ses chiffres.

## 2. Séparer les phrases collées

- Combien d'avis contiennent des phrases collées (« either.The ») ?
- La règle gère-t-elle aussi « wow...Great » ou « down!!By » ?
- La correction donne-t-elle le bon résultat sur des exemples réels ?

<details>
<summary>📖 Comprendre</summary>

**D'où vient le problème ?** Les retours à la ligne ont été supprimés à la création du dataset
(constaté dans l'EDA, section 9). Deux phrases se retrouvent collées : « …is a snap.But not… ».

**Pourquoi c'est gênant ?** Un découpage en mots sur les espaces voit « snap.But » comme un seul
mot, qui n'existe nulle part ailleurs : le modèle ne peut rien en apprendre. En remettant une
espace, on retrouve « snap. » et « But », deux mots connus.

**La règle** (une expression régulière) : une lettre **minuscule**, puis une ou plusieurs
ponctuations de fin de phrase (`.`, `!`, `?`), puis directement une **majuscule**. On ajoute une
espace après la ponctuation. Exiger une minuscule avant évite de toucher aux sigles comme « U.S.A ».

**Limite connue** : dans une adresse web comme « …id=1038.Verizon », la règle ajoute une espace.
Ce n'est pas grave ici : les liens n'apportent rien au sentiment.

</details>

In [ ]:
# Une minuscule, une ou plusieurs ponctuations de fin, puis directement une majuscule.
GLUED_SENTENCES = re.compile(r"(?<=[a-z])[.!?]+(?=[A-Z])")

for name, frame in [("train", train), ("test", test)]:
    n_glued = frame["content"].str.contains(GLUED_SENTENCES).sum()
    print(f"{name:5} : {n_glued} avis concernés ({n_glued / len(frame):.1%})")

In [ ]:
def fix_glued_sentences(text: str) -> str:
    """Ajoute une espace entre deux phrases collées (« snap.But » devient « snap. But »)."""
    return GLUED_SENTENCES.sub(r"\g<0> ", text)  # \g<0> : la ponctuation trouvée


# Quelques exemples réels du train, avant et après correction.
glued = train.loc[train["content"].str.contains(GLUED_SENTENCES), "content"]
for text in glued.sample(5, random_state=0):
    match = GLUED_SENTENCES.search(text)
    excerpt = text[max(match.start() - 25, 0) : match.end() + 25]
    print("avant :", excerpt)
    print("après :", fix_glued_sentences(excerpt), end="\n\n")

In [ ]:
for name, frame in [("train", train), ("test", test)]:
    before = frame["content"].copy()
    for column in ["title", "content"]:
        frame[column] = frame[column].map(fix_glued_sentences)
    changed = (frame["content"] != before).sum()
    log_step("phrases collées", name, len(frame), len(frame), changed)

pd.DataFrame(cleaning_log)

**Observations**

- **11 637 avis du train (23,3 %)** et **2 343 du test (23,4 %)** contiennent des phrases collées.
  C'est plus que les 10 749 de l'EDA : la règle gère aussi les ponctuations répétées
  (« leaves.... boring!Women's »), que la recherche de l'EDA ne voyait pas.
- Sur les 5 exemples tirés au hasard, la correction est juste à chaque fois (« same.I » devient
  « same. I »).
- Les deux jeux sont touchés dans la même proportion : la règle se comporte pareil sur le test.
- Les titres sont corrigés avec la même règle.

## 3. Nettoyer les restes de HTML

- Quelles « balises » trouve-t-on vraiment dans les avis ? Sont-elles toutes du HTML ?
- Quelles entités HTML (`&amp;`, `&#146;`…) apparaissent ?
- Que garder, que retirer ?

<details>
<summary>📖 Comprendre</summary>

**Deux sortes de restes HTML** :
- les **entités** : des codes qui remplacent un caractère (`&amp;` → `&`, `&quot;` → `"`,
  `&#146;` → `'`). On les **décode** pour retrouver le vrai caractère ;
- les **balises** : des instructions de mise en page (`<i>`, `<br />`, `<a href=…>`). Elles ne
  font pas partie du texte écrit par le client : on les **retire**.

**Le piège de l'EDA.** L'EDA comptait comme « balise » tout ce qui est entre chevrons. En
regardant de près, beaucoup n'en sont pas : `<sigh>`, `<wink>`, `<just kidding>` sont **écrits
par le client** pour exprimer une émotion, voire signaler de l'ironie ! Les retirer ferait perdre
du sens. La règle ne retire donc que les **vraies balises HTML**, listées explicitement.

**Le caractère « � »** apparaît quand un texte a été mal encodé : il ne veut plus rien dire,
on le supprime.

</details>

In [ ]:
# Tout ce qui ressemble à une balise : un chevron, une lettre, du texte, un chevron fermant.
train["content"].str.extractall(r"(</?[a-zA-Z][^<>]{0,80}>)")[0].value_counts()

In [ ]:
train["content"].str.extractall(r"(&#?\w+;)")[0].value_counts()

In [ ]:
# Seules les vraies balises HTML sont retirées : « <sigh> » ou « <just kidding> » sont écrits par
# le client et portent du sens (émotion, ironie), on les garde.
HTML_TAG = re.compile(r"</?(?:a|b|i|u|p|br|em|strong|div|span|font)\b[^>]*>", re.IGNORECASE)
REPLACEMENT_CHAR = "\ufffd"  # « � » : caractère illisible laissé par une erreur d'encodage


def clean_html(text: str) -> str:
    """Décode les entités HTML, retire les balises HTML et les caractères illisibles."""
    text = html.unescape(text).replace(REPLACEMENT_CHAR, "")
    return HTML_TAG.sub(" ", text)


for name, frame in [("train", train), ("test", test)]:
    before = frame["content"].copy()
    for column in ["title", "content"]:
        frame[column] = frame[column].map(clean_html)
    changed = (frame["content"] != before).sum()
    log_step("HTML", name, len(frame), len(frame), changed)

pd.DataFrame(cleaning_log)

In [ ]:
# Vérification : il ne reste aucune vraie balise, et les marqueurs d'émotion sont toujours là.
print("vraies balises restantes :", train["content"].str.contains(HTML_TAG).sum())
print("« <sigh> » toujours présent :", train["content"].str.contains("<sigh>", regex=False).sum())

**Observations**

- **Vraies balises HTML**, retirées : `<br />` (4), `<i>` et `</i>` (3 chacune), `<u>` et `</u>`
  (2 chacune), deux liens `<a …>` et leurs `</a>`.
- **Faux amis gardés** : `<sigh>`, `<scroll>`, `<paste>` (2 chacun), `<just kidding>`, `<wink>`,
  `<space>`, `<Secret Honor>`, `<Carmina Burana>`… Ils sont écrits par les clients ; les retirer
  aurait fait perdre des indices d'émotion et d'ironie. La vérification confirme : 0 vraie balise
  restante, les 2 `<sigh>` toujours là.
- **Vraies entités**, décodées : `&#65533;` (8, le caractère illisible « � », supprimé),
  `&#146;` (4, une apostrophe), `&quot;` (2), des ponctuations « pleine largeur » (`&#65281;`…).
- **Fausses entités**, laissées telles quelles : `&D;` (18), `&B;` (15), `&W;` (5)… Ce sont des
  abréviations comme « D&D », « R&B », « b&w » après lesquelles la source a ajouté un `;`.
  Inoffensives pour le sens, elles ne sont pas corrigées.
- Au total, le nettoyage HTML modifie **16 avis du train** et **1 du test** : un problème très rare.

## 4. Retirer les avis non anglais (et les textes parasites)

- Combien d'avis l'heuristique de l'EDA signale-t-elle dans le train et le test ?
- Parmi ceux du train, lesquels sont vraiment dans une autre langue ?
- Que retire la règle à tort, et est-ce acceptable ?

<details>
<summary>📖 Comprendre</summary>

**Pourquoi les retirer ?** Notre modèle apprendra le sentiment à partir de mots **anglais**.
Quelques dizaines d'avis en espagnol n'apprendraient rien d'utile et brouilleraient le vocabulaire.
En production, on ferait pareil : on filtrerait les avis non anglais **avant** de les classer.

**L'heuristique de l'EDA** : un avis en anglais contient forcément des mots très courants
(« the », « and », « is »…). Si moins de **5 %** de ses mots en font partie, il est signalé.

**Une règle imparfaite, mais mesurée.** Elle attrape aussi quelques avis anglais très courts ou
atypiques (« Boring boring boring movie »), qui ne contiennent aucun de ces mots courants. Plutôt
que de compliquer la règle, on **lit** ce qu'elle retire (cellule ci-dessous) et on décide si la
perte est acceptable. C'est la bonne pratique avec une heuristique : mesurer ses erreurs.

**Pour le test**, on applique exactement la même règle, et on ne regarde que le nombre d'avis retirés.

</details>

In [ ]:
COMMON_ENGLISH_TEXT = "the and a to of is it i this in for that was not with but my you on"
COMMON_ENGLISH = set(COMMON_ENGLISH_TEXT.split())
ENGLISH_SHARE_MIN = 0.05


def english_share(text: str) -> float:
    """Part des mots du texte qui sont des mots anglais très courants."""
    words = re.findall(r"[a-z']+", text.lower())
    return sum(word in COMMON_ENGLISH for word in words) / max(len(words), 1)


for name, frame in [("train", train), ("test", test)]:
    flagged = frame["content"].map(english_share) < ENGLISH_SHARE_MIN
    print(f"{name:5} : {flagged.sum()} avis signalés ({flagged.mean():.2%})")

In [ ]:
# Signalés du train : quelle langue ? On compte quelques mots très courants de chaque langue.
LANGUAGE_HINTS_TEXT = {
    "espagnol": "de la que el muy es y en los las con para una por lo no me",
    "français": "le les des est et une pour dans pas je ce il qui sur très du",
}
LANGUAGE_HINTS = {lang: set(words.split()) for lang, words in LANGUAGE_HINTS_TEXT.items()}


def guess_language(text: str) -> str:
    words = re.findall(r"[a-zà-ÿ']+", text.lower())
    scores = {lang: sum(w in hints for w in words) for lang, hints in LANGUAGE_HINTS.items()}
    best = max(scores, key=lambda lang: scores[lang])
    return best if scores[best] >= 3 else "autre"


flagged_train = train[train["content"].map(english_share) < ENGLISH_SHARE_MIN]
flagged_language = flagged_train["content"].map(guess_language)
flagged_language.value_counts()

In [ ]:
# Les « autre » sont lus un par un : autre langue, texte parasite, ou avis anglais retiré à tort ?
flagged_train.loc[flagged_language == "autre", ["label", "content"]]

In [ ]:
for name in ["train", "test"]:
    frame = train if name == "train" else test
    kept = frame[frame["content"].map(english_share) >= ENGLISH_SHARE_MIN]
    log_step("avis non anglais", name, len(frame), len(kept), 0)
    if name == "train":
        train = kept
    else:
        test = kept

pd.DataFrame(cleaning_log)

**Observations**

- L'heuristique signale **114 avis du train (0,23 %)** et **25 du test (0,25 %)**.
- Dans le train, après lecture (celle de l'EDA pour les 90 « espagnol » et « français », celle
  du tableau ci-dessus pour les 24 « autre ») :
  - **93 avis ne sont pas en anglais** : 80 en espagnol, 9 en français, 1 en catalan, 1 en
    polonais [14491], 1 en suédois [15272], 1 en portugais [26431] ;
  - **4 textes parasites** : une fiche technique de Blu-ray [17787], des résultats de catch [19458],
    l'adresse postale berlinoise de l'EDA [22598], « iiii llll oooo » [26922] ;
  - **17 avis anglais retirés à tort** : courts ou atypiques (« Boring boring boring movie. »
    [48112], « DIS ALBOOM IZ SOO HEEVY… » [39470]…), dont **9 négatifs** et 8 positifs.
- **La perte est acceptable** : 17 avis sur 50 000 (0,03 %). Côté besoin, 9 avis négatifs sur
  environ 25 000 (0,04 %) : sans effet mesurable sur un objectif de 90 % de rappel.
- **À retenir pour la production** : le même filtre y écarterait aussi quelques avis anglais très
  courts, qui ne seraient donc jamais signalés. C'est un risque faible, noté dans le suivi.
- L'adresse postale de l'EDA disparaît avec ce filtre : il n'y a plus besoin de la traiter à part.

## 5. Masquer les données personnelles

- Trouve-t-on des e-mails, des numéros de téléphone, des adresses postales ?
- Les motifs de recherche attrapent-ils de vraies données personnelles, ou des faux positifs ?
- Faut-il supprimer les avis concernés, ou seulement masquer la donnée ?

<details>
<summary>📖 Comprendre</summary>

**Pourquoi s'en soucier ?** Un e-mail ou un numéro de téléphone permet de contacter une personne.
Ces informations n'aident en rien à deviner le sentiment, et un jeu d'entraînement ne doit pas
les diffuser (le RGPD encadre leur usage). L'EDA avait trouvé une adresse postale ; la recherche
est ici **systématique**, dans le train **et** le test.

**Masquer plutôt que supprimer.** Remplacer l'e-mail par `[EMAIL]` retire la donnée personnelle
mais garde l'avis, qui reste utile pour apprendre. Supprimer l'avis entier perdrait un exemple
pour rien.

**Tester un motif avant de l'adopter.** Un motif « 5 chiffres + un mot en majuscule » pour
repérer « 13353 Berlin » attrape aussi « 21277 Kennel » (une référence de produit) : c'est un
**faux positif**. On regarde donc ce que chaque motif trouve avant de décider de l'utiliser.

**Pas d'affichage des données trouvées** : on n'imprime que des comptes, des noms de domaine
(`hotmail.com`) ou des extraits autour d'une donnée déjà masquée. Sinon, le notebook lui-même
diffuserait les données qu'on cherche à retirer.

</details>

In [ ]:
EMAIL = re.compile(r"[\w.+-]+@[\w-]+\.[a-zA-Z]{2,}")
PHONE = re.compile(r"\(?\b\d{3}\)?[-.\s]\d{3}[-.\s]\d{4}\b")
# Deux motifs d'adresse postale, testés ci-dessous avant d'être adoptés ou écartés.
STREET = re.compile(
    r"\b\d{1,5}\s+(?:[A-Z][a-z]+\s+){1,3}"
    r"(?:Street|St\.|Avenue|Ave\.?|Road|Rd\.|Boulevard|Blvd|Drive|Dr\.|Lane|Ln\.|Court|Ct\.)(?=\W)"
)
ZIP_CITY = re.compile(r"\b\d{5}\s+[A-Z][a-z]+\b")
PATTERNS = {"e-mail": EMAIL, "téléphone": PHONE, "rue": STREET, "code postal + ville": ZIP_CITY}


def full_text(frame: pd.DataFrame) -> pd.Series:
    return frame["title"] + " " + frame["content"]


pd.DataFrame(
    {
        name: {
            split: full_text(frame).str.contains(pattern).sum()
            for split, frame in [("train", train), ("test", test)]
        }
        for name, pattern in PATTERNS.items()
    }
)

In [ ]:
# Ce que trouvent les motifs d'adresse dans le train : vraies adresses ou faux positifs ?
for name, pattern in [("rue", STREET), ("code postal + ville", ZIP_CITY)]:
    print(f"--- {name}")
    for text in train["content"][train["content"].str.contains(pattern)]:
        print(f"  « {pattern.search(text).group(0)} » dans : {text[:70]}…")

In [ ]:
# Pour les e-mails, on n'affiche que le nom de domaine, jamais l'adresse complète.
emails = full_text(train).str.findall(EMAIL).explode().dropna()
emails.str.split("@").str[1].str.lower().value_counts()

In [ ]:
def mask_personal_data(text: str) -> str:
    """Remplace les e-mails et les numéros de téléphone par un repère neutre."""
    return PHONE.sub("[PHONE]", EMAIL.sub("[EMAIL]", text))


for name, frame in [("train", train), ("test", test)]:
    before = full_text(frame)
    for column in ["title", "content"]:
        frame[column] = frame[column].map(mask_personal_data)
    changed = (full_text(frame) != before).sum()
    log_step("données personnelles", name, len(frame), len(frame), changed)

pd.DataFrame(cleaning_log)

In [ ]:
# Vérification : plus aucun e-mail ni téléphone, et un exemple de masquage.
for name, frame in [("train", train), ("test", test)]:
    remaining = full_text(frame).str.contains(EMAIL) | full_text(frame).str.contains(PHONE)
    print(f"{name:5} : {remaining.sum()} avis avec un e-mail ou un téléphone restant")

masked = train.loc[train["content"].str.contains("[EMAIL]", regex=False), "content"].iloc[0]
start = masked.index("[EMAIL]")
print("exemple :", masked[max(start - 60, 0) : start + 20])

**Observations**

- **E-mails** : 40 avis du train et 10 du test. Ce sont surtout des adresses personnelles
  (yahoo.com 10, aol.com 5, hotmail.com 4…) : de vraies données personnelles.
- **Téléphones** : 12 avis du train et 1 du test (surtout des numéros de service client, mais on
  ne peut pas trier au cas par cas) : tous masqués.
- **Motifs d'adresse écartés** : ils ne trouvent que des **faux positifs**. « Rue » trouve
  « 1975 Earls Court » (une salle de concert) et l'adresse d'une association ; « code postal +
  ville » trouve 14 avis du train, tous lus (texte et titres) : uniquement des références de
  produit (« 21277 Kennel », « Hamilton Beach 22708 Toastation »…) ou des nombres, dont deux
  créés par la jonction du titre et du texte (« Hidden Hitch 70530 » + « Was pleased… »).
  Aucune adresse de particulier : ces deux motifs ne sont pas utilisés.
- **Masquage** : 51 avis du train et 11 du test modifiés (titre ou contenu). Vérification :
  plus aucun e-mail ni téléphone dans les deux jeux, et l'avis reste lisible
  (« …email me at [EMAIL]....thanx »).
- **Limite** : 4 adresses sont collées au mot suivant (« optonline.netsincirely »,
  « msn.comcontact »… dans la liste des domaines) ; le masquage emporte aussi ce mot.
  Perte négligeable.
- Non traité : les **noms de personnes** (auteurs, artistes, clients) ne sont pas recherchés ;
  la plupart sont des noms publics (un auteur de livre), indissociables du sens de l'avis.

## 6. Retirer les doublons

- Reste-t-il des avis en double dans le train ? Dans le test ?
- Et des quasi-doublons (identiques à la ponctuation et aux majuscules près) ?
- Un avis du test se retrouve-t-il dans le train (fuite de données) ?

<details>
<summary>📖 Comprendre</summary>

**Pourquoi après le nettoyage ?** Deux avis qui ne différaient que par une entité HTML ou une
phrase collée deviennent identiques une fois nettoyés. On cherche donc les doublons **à la fin**.

**Doublon et quasi-doublon.** Pour comparer, on met chaque texte en minuscules et on retire tout
ce qui n'est ni une lettre ni un chiffre : « Great CD! » et « great cd » deviennent tous deux
« greatcd ». Deux avis égaux après cette transformation sont des **quasi-doublons**.

**La fuite de données** est le cas le plus grave : si un avis du test est aussi dans le train,
le modèle aura « vu la réponse » et sa note finale sera trop belle. L'EDA n'avait vérifié que les
doublons exacts ; on vérifie ici aussi les quasi-doublons.

</details>

In [ ]:
def comparison_key(text: str) -> str:
    """Texte en minuscules, sans ponctuation ni espaces : sert à repérer les quasi-doublons."""
    return re.sub(r"\W", "", text.lower())


train_keys = train["content"].map(comparison_key)
test_keys = test["content"].map(comparison_key)
print(
    "doublons exacts      — train :",
    train.duplicated(subset="content").sum(),
    "| test :",
    test.duplicated(subset="content").sum(),
)
print(
    "quasi-doublons       — train :",
    train_keys.duplicated().sum(),
    "| test :",
    test_keys.duplicated().sum(),
)
print(
    "fuite train → test   — exacte :",
    train["content"].isin(test["content"]).sum(),
    "| quasi :",
    train_keys.isin(test_keys).sum(),
)

In [ ]:
for name in ["train", "test"]:
    frame = train if name == "train" else test
    kept = frame[~frame["content"].map(comparison_key).duplicated()]
    log_step("doublons", name, len(frame), len(kept), 0)
    if name == "train":
        train = kept
    else:
        test = kept

pd.DataFrame(cleaning_log)

**Observations**

- **1 doublon** dans le train (celui repéré par l'EDA), **aucun** dans le test ; le nettoyage n'en
  a pas créé de nouveau (1 quasi-doublon dans le train, le même).
- **Aucune fuite** entre train et test, ni exacte ni en quasi-doublon : la vérification plus
  stricte que celle de l'EDA le confirme.
- Le doublon du train est retiré : il reste 49 885 avis avant le découpage.

## 7. Mettre de côté la validation

- Combien d'avis garder pour la validation (10 % du train, décidé en phase 2) ?
- La proportion de positifs et de négatifs est-elle la même dans train, validation et test ?

<details>
<summary>📖 Comprendre</summary>

**Pourquoi une validation ?** Pour choisir entre plusieurs modèles, ou régler le **seuil de
décision** (à partir de quelle probabilité on signale un avis), il faut une note. Si on la
calculait sur le test, on finirait par choisir ce qui marche sur **ce** test précis : la note
finale ne serait plus honnête. La validation sert de terrain d'essai ; le test reste intact
jusqu'à la phase 5.

**Découpage stratifié.** Un tirage au hasard pourrait, par malchance, mettre 53 % de positifs
dans la validation et 49 % dans le train. Le découpage **stratifié** garantit la même proportion
de chaque label des deux côtés.

**Graine fixe (42)** : le même découpage à chaque exécution, donc des résultats comparables.

</details>

In [ ]:
rows_before = len(train)
train, val = train_test_split(train, test_size=0.1, stratify=train["label"], random_state=SEED)
log_step("découpage validation", "train", rows_before, len(train), 0)
log_step("découpage validation", "validation", 0, len(val), 0)

splits = {"train": train, "validation": val, "test": test}
pd.DataFrame(
    {
        "avis": {name: len(frame) for name, frame in splits.items()},
        "positifs (%)": {
            name: round(frame["label"].mean() * 100, 2) for name, frame in splits.items()
        },
    }
)

**Observations**

- **Train : 44 896 avis**, **validation : 4 989 avis** (10 %), **test : 9 975 avis**.
- Part de positifs : **49,87 %** dans le train comme dans la validation (le découpage stratifié a
  fonctionné), **50,25 %** dans le test. Les trois jeux restent équilibrés.
- La validation contient environ 2 500 avis négatifs : assez pour mesurer le rappel des négatifs
  à environ un point près.

## 8. Contrôles de qualité

- Les trois jeux respectent-ils toutes les règles (colonnes, labels, pas de texte vide, pas de
  doublon, pas de donnée personnelle, pas d'avis non anglais) ?
- Un même avis se trouve-t-il dans deux jeux à la fois ?

<details>
<summary>📖 Comprendre</summary>

**Un contrôle qualité** est une affirmation qui doit **toujours** être vraie sur des données
propres : « tous les labels valent 0 ou 1 », « aucun texte n'est vide »… On les écrit sous forme
de code, pour pouvoir les relancer à chaque fois qu'on recrée les données.

**Pourquoi les écrire alors qu'on vient de nettoyer ?** Parce que le nettoyage peut avoir un bug,
ou que les données brutes peuvent changer un jour. Dans le code du projet, ces contrôles
**arrêteront** la création des données au moindre problème, plutôt que de laisser passer des
données abîmées jusqu'au modèle.

</details>

In [ ]:
EXPECTED_COLUMNS = ["label", "title", "content"]


def quality_problems(frame: pd.DataFrame) -> list[str]:
    """Renvoie la liste des contrôles qui échouent (liste vide : données conformes)."""
    text = full_text(frame)
    checks = {
        "colonnes attendues": list(frame.columns) == EXPECTED_COLUMNS,
        "labels 0 ou 1": bool(frame["label"].isin([0, 1]).all()),
        "aucune valeur manquante": not frame.isna().any().any(),
        "aucun texte vide": not frame["content"].str.strip().eq("").any(),
        "aucun quasi-doublon": not frame["content"].map(comparison_key).duplicated().any(),
        "aucun e-mail": not text.str.contains(EMAIL).any(),
        "aucun téléphone": not text.str.contains(PHONE).any(),
        "que de l'anglais": bool((frame["content"].map(english_share) >= ENGLISH_SHARE_MIN).all()),
    }
    return [name for name, ok in checks.items() if not ok]


{name: quality_problems(frame) or "OK" for name, frame in splits.items()}

In [ ]:
# Aucun avis ne doit se trouver dans deux jeux à la fois.
keys = {name: set(frame["content"].map(comparison_key)) for name, frame in splits.items()}
for first, second in [("train", "validation"), ("train", "test"), ("validation", "test")]:
    print(f"{first} et {second} : {len(keys[first] & keys[second])} avis en commun")

**Observations** : les **trois jeux passent tous les contrôles** (colonnes, labels, aucune valeur
manquante, aucun texte vide, aucun quasi-doublon, aucun e-mail ni téléphone, que de l'anglais), et
**aucun avis n'est partagé** entre deux jeux. Ces mêmes contrôles seront exécutés automatiquement
par le code du projet à chaque création des données.

## 9. Bilan avant / après

- Combien d'avis chaque étape a-t-elle retirés ou modifiés, dans chaque jeu ?
- Combien d'avis reste-t-il au total, et quelle part a-t-on perdue ?

<details>
<summary>📖 Comprendre</summary>

**Le journal** rassemble toutes les étapes : c'est la preuve que chaque ligne perdue s'explique.
Une perte inexpliquée (« on avait 50 000 avis, il en reste 48 000, on ne sait pas pourquoi »)
serait le signe d'un bug.

Dans ce tableau, « textes modifiés » compte les avis dont le contenu a changé sans être retiré
(phrases séparées, HTML nettoyé, données masquées).

</details>

In [ ]:
pd.DataFrame(cleaning_log)

In [ ]:
summary = pd.DataFrame(
    {
        "avant": {"train + validation": len(raw_train), "test": len(raw_test)},
        "après": {"train + validation": len(train) + len(val), "test": len(test)},
    }
)
summary["retirés"] = summary["avant"] - summary["après"]
summary["retirés (%)"] = (summary["retirés"] / summary["avant"] * 100).round(2)
summary

**Observations**

- **Train + validation** : 50 000 → 49 885 avis, soit **115 retirés (0,23 %)** : 114 par le filtre
  de langue, 1 doublon.
- **Test** : 10 000 → 9 975 avis, soit **25 retirés (0,25 %)**, tous par le filtre de langue.
- Chaque ligne perdue s'explique par une étape du journal.
- **Textes modifiés sans être retirés** : 11 637 (phrases collées), 16 (HTML) et 51 (données
  personnelles) dans le train ; 2 343, 1 et 11 dans le test.
- Le nettoyage **corrige beaucoup** (près d'un avis sur quatre) mais **retire très peu** : les
  données de départ étaient déjà de bonne qualité, comme l'avait montré l'EDA.

## 10. Conclusions

- Quelles règles de nettoyage retient-on, et avec quel effet ?
- Que faut-il porter dans le code du projet (`src/`) ?
- Quelles limites et questions restent ouvertes ?

<details>
<summary>📖 Comprendre</summary>

- Ce notebook sert à **décider et vérifier** les règles. Une fois validé, elles sont recopiées
  dans `src/review_pilot/data/`, testées, et enchaînées dans une seule commande qui recrée les
  trois jeux. Le notebook reste tel quel : c'est la trace du raisonnement.
- Les deux versions (notebook et code du projet) doivent produire **exactement** les mêmes
  données : on le vérifiera en comparant les nombres d'avis de chaque jeu.

</details>

### Synthèse : les règles retenues

| Étape | Constat | Section | Règle retenue |
|---|---|---|---|
| Phrases collées | 23,3 % des avis du train | 2 | ajouter une espace après `.`, `!`, `?` (même répétés) entre une minuscule et une majuscule |
| HTML | 16 avis du train ; beaucoup de « balises » sont des marqueurs d'émotion | 3 | décoder les entités, retirer le caractère « � » et **seulement** les vraies balises HTML |
| Langue | 114 avis signalés, dont 93 non anglais et 4 parasites | 4 | retirer les avis dont moins de 5 % des mots sont des mots anglais courants (17 avis anglais perdus, accepté) |
| Données personnelles | 40 e-mails et 12 téléphones dans le train | 5 | **masquer** (`[EMAIL]`, `[PHONE]`) plutôt que supprimer l'avis ; motifs d'adresse écartés (faux positifs) |
| Doublons | 1 dans le train, aucune fuite | 6 | retirer les quasi-doublons à l'intérieur de chaque jeu |
| Validation | équilibre conservé (49,87 %) | 7 | 10 % du train, découpage stratifié, graine 42 |
| Contrôles | tous au vert, aucun avis partagé | 8 | les exécuter à chaque création des données, et s'arrêter en cas d'échec |

**Écart avec le plan de la phase 2** : le plan prévoyait de **retirer** les avis contenant une
donnée personnelle. Le notebook propose de les **masquer** : la donnée disparaît, mais l'avis reste
utilisable. C'est à valider avec ce notebook.

### Ce qui sera porté dans le code du projet (`src/review_pilot/data/`)

| Fichier | Contenu (repris de ce notebook) |
|---|---|
| `download.py` | l'actuel `review_pilot/dataset.py`, déplacé |
| `clean.py` | `fix_glued_sentences`, `clean_html`, `english_share`, `mask_personal_data`, `comparison_key`, et une fonction qui applique toutes les règles dans l'ordre de ce notebook |
| `split.py` | le découpage stratifié train / validation |
| `quality.py` | `quality_problems` et le contrôle « aucun avis partagé entre deux jeux » |
| `build.py` | la commande unique : `data/raw/` → `data/processed/{train,val,test}.parquet` + le rapport de qualité |

Le code du projet devra produire **exactement** les mêmes nombres d'avis : 44 896 (train),
4 989 (validation), 9 975 (test). Ce notebook reste tel quel après le portage : son code est
volontairement dupliqué dans `src/`, où il sera testé.

### Questions encore ouvertes

- Les **fausses entités** (« R&B; », « D&D; ») sont laissées telles quelles : faut-il retirer ce `;`
  parasite ? Sans enjeu pour le sentiment, non traité.
- Les **noms de personnes** ne sont pas recherchés : acceptable pour un dataset public d'avis,
  à revoir si le projet traite un jour de vrais avis clients.
- Le filtre de langue écarte quelques **avis anglais très courts** : en production, ils ne
  seraient jamais signalés. Faut-il les rattraper (par exemple en ne filtrant que les avis de plus
  de 10 mots) ? À surveiller lors de l'analyse d'erreurs (phase 6).